# Five Whys with LangGraph

Anton Antonov   
March 2026

---

## Introduction


This notebook builds a cyclical **LangGraph** for root cause analysis using the **Five Whys** method.

The graph takes:
- a problem description document
- number of iterations (how many whys)

State fields:
- problem context
- effective problem formulation
- current answer
- current iteration


### Prompt for code generation

One way to generate code similar to the one is this notebook is to use following prompt:

```text
Make Jupyter notebook "./docs/Five-Whys-LLM-graph.ipynb" that implements an LLM graph using the Python library "LangGraph" for the root cause analysis method "Five Whys".
The graph is cyclical and takes as arguments a problem description document and a number of iterations -- those are the number of whys.
It should have nodes for:

1. Effective problem formulation
2. Check has the root cause been found or not
3. Asking and answering the next why from the current answer

The graph should keep a state with the elements:

- Problem context
- Effective problem formulation
- Current answer
- Current iteration (number of the current "why")
```

---

## Setup

In [ ]:
# If needed, install dependencies:
# %pip install -q langgraph langchain-openai

import os
from typing import TypedDict, Optional

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

from IPython.display import Markdown, display

In [ ]:
llm = ChatOpenAI(model="gpt-5-mini")

----

## LLM graph

In [ ]:
class FiveWhysState(TypedDict, total=False):
    # Required by the prompt
    problem_context: str
    effective_problem_formulation: str
    current_answer: str
    current_iteration: int

    # Control fields for graph execution
    max_iterations: int
    root_cause_found: bool
    why_question: str
    why_answer_history: list[dict]


In [ ]:
def effective_problem_formulation_node(state: FiveWhysState) -> FiveWhysState:
    """Node 1: Build an effective problem formulation from the problem context."""
    prompt = f"""
You are a root-cause analysis assistant.

Given this problem context/document, write an effective, concise problem formulation
for Five Whys analysis (clear symptom, scope, and impact).

Problem context:
{state['problem_context']}

Return only the formulation text.
"""
    formulation = llm.invoke(prompt).content.strip()

    return {
        **state,
        "effective_problem_formulation": formulation,
        "current_answer": formulation,
        "current_iteration": 0,
        "root_cause_found": False,
        "why_answer_history": [],
    }


def ask_and_answer_next_why_node(state: FiveWhysState) -> FiveWhysState:
    """Node 3: Ask and answer the next why based on current answer."""
    next_iteration = state["current_iteration"] + 1

    prompt = f"""
You are performing Five Whys analysis.

Problem formulation:
{state['effective_problem_formulation']}

Current answer (latest cause statement):
{state['current_answer']}

Generate:
1) The next why-question for iteration {next_iteration}
2) A plausible answer to that why-question grounded in the context

Output format exactly:
WHY: <question>
ANSWER: <answer>
"""

    result = llm.invoke(prompt).content.strip().splitlines()
    why_line = next((line for line in result if line.startswith("WHY:")), "WHY: Why?")
    answer_line = next((line for line in result if line.startswith("ANSWER:")), "ANSWER: Unknown")

    why_question = why_line.replace("WHY:", "", 1).strip()
    next_answer = answer_line.replace("ANSWER:", "", 1).strip()
    history = state.get("why_answer_history", [])
    history.append({
        "iteration": next_iteration,
        "why": why_question,
        "answer": next_answer,
    })

    return {
        **state,
        "why_question": why_question,
        "current_answer": next_answer,
        "current_iteration": next_iteration,
        "why_answer_history": history,
    }


def check_root_cause_found_node(state: FiveWhysState) -> FiveWhysState:
    """Node 2: Decide if root cause has been found."""
    prompt = f"""
You are checking whether the current cause statement is a root cause.

Problem formulation:
{state['effective_problem_formulation']}

Current cause statement (iteration {state['current_iteration']}):
{state['current_answer']}

Answer ONLY YES or NO.
YES = sufficiently fundamental and actionable root cause.
NO = still superficial/intermediate.
"""

    decision = llm.invoke(prompt).content.strip().upper()
    root_found = decision.startswith("YES")

    return {
        **state,
        "root_cause_found": root_found,
    }


In [ ]:
def route_after_check(state: FiveWhysState) -> str:
    # Stop if root cause found or iteration limit reached; otherwise continue cycle.
    if state.get("root_cause_found", False):
        return END
    if state["current_iteration"] >= state["max_iterations"]:
        return END
    return "ask_next_why"


builder = StateGraph(FiveWhysState)

# Required nodes
builder.add_node("effective_problem_formulation", effective_problem_formulation_node)
builder.add_node("check_root_cause", check_root_cause_found_node)
builder.add_node("ask_next_why", ask_and_answer_next_why_node)

# Graph shape: START -> formulation -> ask why -> check -> (END or ask why)
builder.add_edge(START, "effective_problem_formulation")
builder.add_edge("effective_problem_formulation", "ask_next_why")
builder.add_edge("ask_next_why", "check_root_cause")
builder.add_conditional_edges("check_root_cause", route_after_check)

five_whys_graph = builder.compile()


In [ ]:
five_whys_graph

In [ ]:
def run_five_whys(problem_document: str, num_iterations: int = 5) -> FiveWhysState:
    initial_state: FiveWhysState = {
        "problem_context": problem_document,
        "effective_problem_formulation": "",
        "current_answer": "",
        "current_iteration": 0,
        "max_iterations": num_iterations,
        "root_cause_found": False,
        "why_question": "",
        "why_answer_history": [],
    }
    return five_whys_graph.invoke(initial_state)


---

## Data

In [ ]:
problem_document = """
Customer support tickets increased by 40% in the last month.
Most complaints are about delayed order confirmations and wrong shipping status.
The issue started after the latest backend release.
"""

In [ ]:
with open('your/file', 'r') as file:
    problem_document = file.read()
print(len(problem_document))

-----

## Experiments

In [ ]:
result = run_five_whys(problem_document, num_iterations=5)

In [ ]:
def create_markdown_document(result):
    sections = []
    sections.append(f"## Problem context:\n{result['problem_context']}")
    sections.append(f"## Effective problem formulation:\n{result['effective_problem_formulation']}")
    sections.append(f"## Current iteration:\n{result['current_iteration']}")
    sections.append(f"## Root cause found:\n{result['root_cause_found']}")
    sections.append(f"## Latest why question:\n{result.get('why_question', '')}")
    sections.append(f"## Current answer (latest cause):\n{result['current_answer']}")
    history = result.get("why_answer_history", [])
    history_md = "## Accumulated WHY/ANSWER history:"
    for item in history:
        history_md += f"\n\n### Why #{item['iteration']}:\n**WHY:** {item['why']}\n\n**ANSWER:** {item['answer']}"
    sections.append(history_md)
    return "\n\n---\n\n".join(sections)

In [ ]:
# Example usage:
markdown_doc = create_markdown_document(result)
display(Markdown(markdown_doc))